In [ ]:
# Clear CUDA cache and restart
import torch
torch.cuda.empty_cache()

# If error persists, restart runtime: Runtime -> Restart Runtime
# Then run this:

import gc
gc.collect()
torch.cuda.empty_cache()

print("CUDA cache cleared")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

class SUIMDataset(Dataset):
    def __init__(self, root_dir, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.target_transform = target_transform

        self.image_paths = sorted(glob.glob(os.path.join(root_dir, "images", "*.jpg")))
        self.mask_paths = sorted(glob.glob(os.path.join(root_dir, "masks", "*.png")))

        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {root_dir}/images")
        if len(self.mask_paths) == 0:
            raise ValueError(f"No masks found in {root_dir}/masks")

        print(f"Found {len(self.image_paths)} images")
        print(f"Found {len(self.mask_paths)} masks")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            mask = self.target_transform(mask)

        if not isinstance(mask, torch.Tensor):
            mask = transforms.ToTensor()(mask)


        mask = (mask * 255).long().squeeze(0)
        mask = remap_mask(mask)

        return image, mask

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

In [ ]:

dataset_path = os.path.join(path, "dataset")
dataset = SUIMDataset(dataset_path, transform=image_transform, target_transform=mask_transform)

from sklearn.model_selection import train_test_split
indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

from torch.utils.data import Subset
train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
import numpy as np
def visualize_samples(dataset, num_samples=4):
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, num_samples*3))

    for i in range(num_samples):
        image, mask = dataset[i]

        image_display = image.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image_display = std * image_display + mean
        image_display = np.clip(image_display, 0, 1)

        axes[i, 0].imshow(image_display)
        axes[i, 0].set_title(f'Image {i+1}')
        axes[i, 0].axis('off')

        mask_display = mask.numpy() if isinstance(mask, torch.Tensor) else mask
        axes[i, 1].imshow(mask_display, cmap='tab20')
        axes[i, 1].set_title(f'Mask {i+1} (Classes: {len(np.unique(mask_display))})')
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

visualize_samples(dataset, num_samples=4)

In [ ]:
# TO DO
# Install segmentation_models_pytorch if not already installed
!pip install segmentation-models-pytorch -q

import segmentation_models_pytorch as smp
import torch
import torch.nn as nn

In [ ]:
class UNetModel(nn.Module):
    def __init__(self, num_classes, encoder_name='efficientnet-b1', encoder_weights='imagenet'):
        super(UNetModel, self).__init__()

        self.model = smp.Unet(
            encoder_name=encoder_name,        # Use EfficientNet-B1 as encoder
            encoder_weights=encoder_weights,  # Use pretrained ImageNet weights
            in_channels=3,                    # RGB input
            classes=num_classes,              # Number of segmentation classes
            activation=None                   # No activation (we'll use softmax/cross-entropy)
        )

    def forward(self, x):
        return self.model(x)

# Determine number of classes from the dataset
sample_image, sample_mask = dataset[0]
num_classes = len(torch.unique(sample_mask))
print(f"Number of segmentation classes: {num_classes}")

# Initialize the model
model = UNetModel(num_classes=num_classes, encoder_name='efficientnet-b1', encoder_weights='imagenet')

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"\nModel created successfully!")
print(f"Using device: {device}")
print(f"\nModel architecture:")
print(model)

In [ ]:
# TO DO
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train the model for one epoch"""
    model.train()
    running_loss = 0.0

    for images, masks in tqdm(train_loader, desc="Training", leave=False):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)
    return epoch_loss

In [ ]:
def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc="Validation", leave=False):
            images = images.to(device)
            masks = masks.to(device)


            outputs = model(images)

            loss = criterion(outputs, masks)
            running_loss += loss.item()

    epoch_loss = running_loss / len(val_loader)
    return epoch_loss

def calculate_iou(pred_mask, true_mask, num_classes):
    ious = []
    pred_mask = pred_mask.cpu().numpy()
    true_mask = true_mask.cpu().numpy()

    for cls in range(num_classes):
        pred_inds = pred_mask == cls
        target_inds = true_mask == cls
        intersection = (pred_inds & target_inds).sum()
        union = (pred_inds | target_inds).sum()

        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)

    return np.nanmean(ious)

In [ ]:
# TO DO
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function: {criterion}")
print(f"Optimizer: {optimizer}")

In [ ]:
# Define training and validation functions
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    return running_loss / len(train_loader)

def val_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

    return running_loss / len(val_loader)

In [ ]:
model = UNetModel(num_classes=num_classes, encoder_name='efficientnet-b1', encoder_weights='imagenet')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 25
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)

    val_loss = val_epoch(model, val_loader, criterion, device)
    val_losses.append(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Plot loss curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training Loss', marker='o', linewidth=2)
plt.plot(val_losses, label='Validation Loss', marker='s', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Training Loss', marker='o', linewidth=2)
plt.plot(val_losses, label='Validation Loss', marker='s', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss (Log Scale)', fontsize=14, fontweight='bold')
plt.yscale('log')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=6):
    model.eval()

    fig, axes = plt.subplots(num_samples, 3, figsize=(15, num_samples*3))

    indices = np.random.choice(len(dataset), num_samples, replace=False)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, true_mask = dataset[idx]

            image_input = image.unsqueeze(0).to(device)

            output = model(image_input)
            pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu()

            image_display = image.cpu().numpy().transpose(1, 2, 0)
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            image_display = std * image_display + mean
            image_display = np.clip(image_display, 0, 1)

            axes[i, 0].imshow(image_display)
            axes[i, 0].set_title('Input Image', fontsize=12, fontweight='bold')
            axes[i, 0].axis('off')

            true_mask_display = true_mask.cpu().numpy() if isinstance(true_mask, torch.Tensor) else true_mask
            axes[i, 1].imshow(true_mask_display, cmap='tab20')
            axes[i, 1].set_title('Ground Truth Mask', fontsize=12, fontweight='bold')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(pred_mask.numpy(), cmap='tab20')
            axes[i, 2].set_title('Predicted Mask', fontsize=12, fontweight='bold')
            axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()